In [1]:
import sys
from pathlib import Path
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
sys.path.append("..")
from vacant_lot.data_utils import load_gdb

REPO_ROOT = Path.cwd().parents[0]
GDB_PATH  = REPO_ROOT / "data/nyc_mappluto_22v3_arc_fgdb/MapPLUTO22v3.gdb"
LAYER     = "MapPLUTO_22v3_clipped"
OUT_PATH  = REPO_ROOT / "EDA/outputs/mappluto_22/vacant_lots_map.tif"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [2]:
gdf = load_gdb(GDB_PATH, layer=LAYER)
assert gdf.crs.is_projected, f"Expected projected CRS, got: {gdf.crs}"
print(f"CRS: {gdf.crs}  ({len(gdf):,} parcels)")

CRS: EPSG:2263  (856,998 parcels)


In [3]:
RESOLUTION = 10  # feet (EPSG:2263 units); ~3m — good city-scale detail

minx, miny, maxx, maxy = gdf.total_bounds
width  = int((maxx - minx) / RESOLUTION)
height = int((maxy - miny) / RESOLUTION)
transform = from_bounds(minx, miny, maxx, maxy, width, height)

# Build (geometry, class_value) pairs — vacant drawn last so it overrides
shapes = (
    [(geom, 1) for geom in gdf[gdf["LandUse"] != "11"].geometry if geom is not None]
  + [(geom, 2) for geom in gdf[gdf["LandUse"] == "11"].geometry  if geom is not None]
)

class_raster = rasterize(
    shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype=np.uint8,
)

# Map class → RGB
PALETTE = np.array([
    [ 26,  26,  46],   # 0 — background (dark navy)
    [155, 107, 155],   # 1 — non-vacant (mauve purple)
    [255, 224,  38],   # 2 — vacant (bright yellow)
], dtype=np.uint8)

rgb = PALETTE[class_raster]  # shape (H, W, 3)

# Write georeferenced GeoTIFF
with rasterio.open(
    OUT_PATH, "w",
    driver="GTiff",
    height=height, width=width,
    count=3,
    dtype=np.uint8,
    crs=gdf.crs,
    transform=transform,
    compress="lzw",
) as dst:
    dst.write(rgb[:, :, 0], 1)
    dst.write(rgb[:, :, 1], 2)
    dst.write(rgb[:, :, 2], 3)

print(f"Written: {OUT_PATH}  ({width}×{height} px, {OUT_PATH.stat().st_size / 1e6:.1f} MB)")

Written: /path/to/Vacant_Lot_Detection/EDA/outputs/mappluto_22/vacant_lots_map.tif  (15420×15276 px, 20.5 MB)
